# Induction Lab: demo

An AI marker that finds learner misconceptions in Grade 11 Physical Sciences exam answers, built and evaluated by a Physical Sciences teacher.

So far it covers **one exam question**: Question 11 (electromagnetic induction, 13 marks) of the DBE Grade 11 Physical Sciences Paper 1, November 2018.

**This demo is free.** It needs no API key and no GPU. It runs the **rule engine** (memo answers, positive marking, number fingerprints) and shows the **evaluation results** of the LLM runs. Running an LLM yourself is described at the end.

Press **Runtime → Run all**, or run the cells one by one.

Code and all results: [github.com/Wityd/High-School-induction-learning-material](https://github.com/Wityd/High-School-induction-learning-material)

In [1]:
# Get the project (a few seconds)
!rm -rf repo && git clone -q https://github.com/Wityd/High-School-induction-learning-material.git repo
import sys; sys.path.insert(0, "repo")

from pipeline import misconceptions as mc
from pipeline.marking import mark_numeric, feedback

cat = mc.load()
Q = "dbe_p1_2018_nov_gr11.11"
print(f"Catalogue loaded: {len(cat['misconceptions'])} misconceptions for electromagnetic induction")

Catalogue loaded: 17 misconceptions for electromagnetic induction


## 1. The question

> A SQUARE induction coil with a side length 3 cm and 400 windings is placed perpendicularly in a uniform magnetic field and then rotated through an angle of 45° in 0,08 s. An emf of 7 V is induced in the coil.
>
> **11.2** Calculate the change in the magnetic flux. (3)
> **11.3** Calculate the magnitude of the magnetic field. (4)

Memo: **ΔΦ = −1,4 × 10⁻³ Wb** and **B = 5,31 T**, with *positive marking from 11.2*: a learner who gets 11.2 wrong but uses that answer correctly in 11.3 still earns the 11.3 marks.

*(Question and memo: DBE, used for educational purposes.)*

## 2. Mark three learners

Each learner gives a final answer for 11.2 and 11.3. The rule engine marks them against the memo and names the likely misconception. When an answer is wrong, the method marks depend on the learner's working, so they are shown as a range.

In [2]:
def mark_learner(name, answer_11_2, answer_11_3):
    r2 = mark_numeric(cat, Q + ".2", answer_11_2, max_marks=3)
    r3 = mark_numeric(cat, Q + ".3", answer_11_3, max_marks=4, source_value=answer_11_2)
    print(f"--- {name}: 11.2 = {answer_11_2}, 11.3 = {answer_11_3}")
    print(feedback([r2, r3]))
    print()

mark_learner("Learner A (all correct)", -0.0014, 5.31)
mark_learner("Learner B (forgot N, then used that answer correctly)", 0.56, 2124)
mark_learner("Learner C (correct 11.2, forgot the starting flux in 11.3)", -0.0014, 2.20)

--- Learner A (all correct): 11.2 = -0.0014, 11.3 = 5.31
11.2: 3/3. Correct.
11.3: 4/4. Correct.

--- Learner B (forgot N, then used that answer correctly): 11.2 = 0.56, 11.3 = 2124
11.2: 0-2/3 (method marks need the learner's working). Likely: Leaves out the number of turns (EMI-08). Teacher tip: Ask the learner to say in words what each symbol stands for before substituting.
11.3: 4/4. Earlier answer was wrong but used correctly here (positive marking).

--- Learner C (correct 11.2, forgot the starting flux in 11.3): 11.2 = -0.0014, 11.3 = 2.2
11.2: 3/3. Correct.
11.3: 0-3/4 (method marks need the learner's working). Likely: Final flux taken as the change in flux (EMI-15). Teacher tip: Before substituting, ask for Φ at the start and Φ at the end as two separate numbers, then subtract.



**What to notice**

- **Learner B** loses marks in 11.2 for forgetting the number of turns (EMI-08), but gets **4/4 in 11.3**: the memo's positive marking means the same mistake is not punished twice.
- **Learner C**'s 2,20 T is a *fingerprint*: it is exactly what you get if you use ΔΦ = BA cos 45° and forget the starting flux (EMI-15). Taking the angle from the coil's plane (EMI-12) gives the same number, which is why the LLM layer reads the learner's working.

## 3. Try your own

Change the two numbers and run the cell. Some answers to try:

| 11.2 | 11.3 | What it is |
|---|---|---|
| `0.21875` | `830` | 11.2: dividing by Δt instead of multiplying. 11.3: that answer used correctly |
| `-0.0014` | `0.1593` | Side of the coil not squared |
| `-0.0014` | `0.0531` | 9 cm² converted as 0,09 m² |
| `-0.0014` | `42` | No known mistake: goes to the teacher |

In [3]:
mark_learner("Your learner", -0.0014, 0.1593)   # change these numbers

--- Your learner: 11.2 = -0.0014, 11.3 = 0.1593
11.2: 3/3. Correct.
11.3: 0-3/4 (method marks need the learner's working). Likely: Area not converted to square metres (EMI-13). Teacher tip: Draw 1 m² as a 100 cm × 100 cm square to show why the factor is 10 000.



## 4. One-word answers

**11.6** The north pole of a bar magnet is pushed into a solenoid. Which pole will be induced at the end facing the magnet? *Write only NORTH or SOUTH.* (Memo: **NORTH**)

In [4]:
for answer in ["NORTH", "SOUTH"]:
    found = mc.diagnose_choice(cat, Q + ".6", answer)
    if not found:
        print(f"{answer}: correct, 1/1")
    for m in found:
        print(f"{answer}: 0/1. Likely {m['label']} ({m['id']}).")
        print(f"   Learners say: {m['learner_says']}")
        print(f"   Teacher tip: {m['remediation']}")

NORTH: correct, 1/1
SOUTH: 0/1. Likely Induced current helps the change (EMI-04).
   Learners say: The coil becomes a south pole and pulls the magnet in.
   Teacher tip: Ask: if the coil pulled the magnet in, would it then speed up by itself forever? Use energy conservation to rule it out.


## 5. One misconception in full

Every misconception in the catalogue has what learners say, the correct idea, why learners think it, and a teaching fix.

In [5]:
m = next(m for m in cat["misconceptions"] if m["id"] == "EMI-17")
for key in ["id", "label", "learner_says", "correct_idea", "why_learners_think_this", "remediation"]:
    print(f"{key:24} {m[key]}")

id                       EMI-17
label                    Moving faster makes the magnet stronger
learner_says             The emf increases because the magnetic field becomes stronger when the coil rotates (or the magnet moves) faster.
correct_idea             The field strength B stays the same. Moving faster makes the same change in flux happen in less time, so the rate of change of flux, and the emf, increase.
why_learners_think_this  They need a cause for the bigger emf and attach it to the most visible quantity, the magnet's strength.
remediation              Ask what the magnet is made of and whether that changed. Then compare ΔΦ and Δt for slow and fast movements.


## 6. How well do LLMs do?

For written working and explanations, an LLM reads the answer and names the misconception, choosing only from the catalogue; code then checks it against the memo and the rules. Two free open-source models were tested on the same 23 teacher-labelled cases. These are the saved results from the repository:

In [6]:
import json, pandas as pd

runs = {"Llama 3.1 8B": "llama3.1-8b_v4_report.json", "Qwen 2.5 14B": "qwen2.5-14b_v1_report.json"}
rows = {}
for name, f in runs.items():
    r = json.load(open(f"repo/evaluation/results/{f}"))
    rows[name] = {
        "Labels exactly right": f"{round(r['exact_id_agreement'] * r['rows'])} of {r['rows']}",
        "Agreement with teacher (kappa)": round(r["kappa_has_misconception"], 2),
        "Marks exactly right": f"{round(r['marks_exact_agreement'] * r['rows'])} of {r['rows']}",
        "Sent to the teacher": r["sent_to_teacher"],
        "Wrong and not flagged (silent errors)": r["silent_errors"],
    }
pd.DataFrame(rows)

,Llama 3.1 8B,Qwen 2.5 14B
Labels exactly right,11 of 23,14 of 23
Agreement with teacher (kappa),0.37,0.65
Marks exactly right,19 of 23,16 of 23
Sent to the teacher,9,8
Wrong and not flagged (silent errors),5,4


**In short:** the larger model diagnoses much better (kappa 0.65 vs 0.37, and fewer false alarms on correct work), but marks more harshly, and neither marks calculations and explanations reliably. So the design is: **the LLM diagnoses, the rules mark where they can, and the teacher checks the rest.**

These are **23 constructed cases**, written to contain known misconceptions. They show what the system can detect, not how it performs on real learners. Every run, including the failures, is described in [`evaluation/results/README.md`](https://github.com/Wityd/High-School-induction-learning-material/blob/main/evaluation/results/README.md).

## Run an LLM yourself

To run Llama or Qwen on the 23 cases (free, needs a Colab T4 GPU and a few minutes), follow [`evaluation/README.md`](https://github.com/Wityd/High-School-induction-learning-material/blob/main/evaluation/README.md).